In [1]:
# 01_generate_synthetic_corpus.ipynb -- Ingest real open-source code (Apache Fineract, pinned commit) as the carrier corpus and inject checksum-valid but synthetic sensitive tokens to retain ground-truth labels.
# Real OSS supplies realistic identifiers/logic; synthetic injection supplies measurable recall ground truth.
# Injected sensitive values are randomly generated and correspond to no real person, account, card, or credential.
# Output paths are intentionally not printed (anonymity).

import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

# --- Reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Pinned upstream source (Apache-2.0) -----------------------------------
REPO_URL = "https://github.com/apache/fineract.git"
COMMIT = "e0ae48b391ab1b31b597e5fcaeee4a4419e1841a"
SUBDIR = "fineract-provider/src/main/java"
N_FILES = 150
DOMAIN_HINTS = ("portfolio", "loanaccount", "savings", "client", "account", "interest", "charge")
MIN_LINES, MAX_LINES = 40, 400

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA = ROOT / "data"
SRC = DATA / "_fineract"   # gitignored working copy of upstream source
DATA.mkdir(parents=True, exist_ok=True)

def _run(args, cwd=None):
    # Run a git command quietly; force UTF-8 decoding so locale codecs (e.g. cp949 on Windows) never crash.
    r = subprocess.run(args, cwd=cwd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
                       text=True, encoding="utf-8", errors="replace")
    if r.returncode != 0:
        raise RuntimeError("git step failed (check network access to GitHub and retry)")

# --- Fetch the pinned commit (idempotent; network needed on first run) -----
marker = SRC / ".pinned_ok"
if not marker.exists():
    if SRC.exists():
        import shutil
        shutil.rmtree(SRC)
    _run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", REPO_URL, str(SRC)])
    _run(["git", "-C", str(SRC), "sparse-checkout", "init", "--cone"])
    _run(["git", "-C", str(SRC), "sparse-checkout", "set", SUBDIR])
    _run(["git", "-C", str(SRC), "fetch", "--depth", "1", "origin", COMMIT])
    _run(["git", "-C", str(SRC), "checkout", COMMIT])
    marker.write_text("ok", encoding="utf-8")

# --- Deterministic file selection ------------------------------------------
base = SRC / SUBDIR
candidates = []
for p in sorted(base.rglob("*.java")):
    try:
        text = p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    n = text.count("\n") + 1
    if MIN_LINES <= n <= MAX_LINES:
        rel = p.relative_to(base).as_posix()
        score = 1 if any(h in rel.lower() for h in DOMAIN_HINTS) else 0
        candidates.append((score, rel, text))
# Prefer domain-relevant files, then sample deterministically.
candidates.sort(key=lambda c: (-c[0], c[1]))
pool = candidates[: max(N_FILES * 3, N_FILES)]
random.shuffle(pool)
selected = pool[:N_FILES]

# --- Synthetic sensitive-token generators (checksum-valid where applicable) ---
def _luhn_total(number_str):
    digits = [int(d) for d in number_str]
    total = sum(digits[-1::-2])
    for d in digits[-2::-2]:
        d2 = d * 2
        total += d2 - 9 if d2 > 9 else d2
    return total % 10

def gen_rrn():
    base12 = [random.randint(0, 9) for _ in range(12)]
    weights = [2, 3, 4, 5, 6, 7, 8, 9, 2, 3, 4, 5]
    check = (11 - (sum(b * w for b, w in zip(base12, weights)) % 11)) % 10
    d = base12 + [check]
    return "".join(map(str, d[:6])) + "-" + "".join(map(str, d[6:]))

def gen_card():
    payload = "".join(str(random.randint(0, 9)) for _ in range(15))
    full = next(payload + str(c) for c in range(10) if _luhn_total(payload + str(c)) == 0)
    return "-".join(full[i:i + 4] for i in range(0, 16, 4))

def gen_account():
    return "".join(str(random.randint(0, 9)) for _ in range(random.randint(11, 14)))

def gen_secret():
    import string
    a = string.ascii_letters + string.digits
    kind = random.choice(["sk", "aws", "ghp", "hex"])
    if kind == "sk":
        return "sk-" + "".join(random.choice(a) for _ in range(40))
    if kind == "aws":
        return "AKIA" + "".join(random.choice(string.ascii_uppercase + string.digits) for _ in range(16))
    if kind == "ghp":
        return "ghp_" + "".join(random.choice(a) for _ in range(36))
    return "".join(random.choice("0123456789abcdef") for _ in range(40))

def gen_email():
    u = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(random.randint(5, 9)))
    return f"{u}@{random.choice(['example.com', 'testbank.co.kr', 'corp.local', 'mail.test'])}"

def gen_phone():
    return "010-" + f"{random.randint(0, 9999):04d}" + "-" + f"{random.randint(0, 9999):04d}"

GENERATORS = {"RRN": gen_rrn, "CARD": gen_card, "ACCOUNT": gen_account,
              "SECRET": gen_secret, "EMAIL": gen_email, "PHONE": gen_phone}

FIXTURE = ("\n\n// --- synthetic test fixture (injected; do not ship) ---\n"
           "// customer rrn: {RRN}\n"
           "private static final String API_KEY = \"{SECRET}\";\n"
           "// settlement account {ACCOUNT}; card {CARD}\n"
           "// contact {EMAIL} {PHONE}\n")

# --- Build corpus: real code + injected synthetic tokens -------------------
corpus, ground_truth = [], []
for i, (_, rel, text) in enumerate(selected):
    vals = {t: GENERATORS[t]() for t in GENERATORS}
    code = text + FIXTURE.format(**vals)
    corpus.append({"file_id": i, "rel_path": rel, "code": code})
    for t, v in vals.items():
        ground_truth.append({"file_id": i, "type": t, "value": v})

with open(DATA / "corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False)
pd.DataFrame(ground_truth).to_csv(DATA / "ground_truth.csv", index=False)
with open(DATA / "meta.json", "w", encoding="utf-8") as f:
    json.dump({"source": "apache/fineract", "commit": COMMIT, "license": "Apache-2.0",
               "n_files": len(corpus), "n_sensitive_tokens": len(ground_truth)}, f, indent=2)

# --- Console summary (counts only, no paths) -------------------------------
total_loc = sum(c["code"].count("\n") + 1 for c in corpus)
print("carrier source: apache/fineract @", COMMIT[:10], "(Apache-2.0)")
print("selected real files:", len(corpus), "| total LOC:", total_loc)
print("injected sensitive tokens:", len(ground_truth))
print(pd.DataFrame(ground_truth)["type"].value_counts().to_string())


carrier source: apache/fineract @ e0ae48b391 (Apache-2.0)
selected real files: 150 | total LOC: 14379
injected sensitive tokens: 900
type
RRN        150
CARD       150
ACCOUNT    150
SECRET     150
EMAIL      150
PHONE      150
